In [122]:
import pandas as pd

tweets_trainset = pd.read_csv('data/TweetsTrainset.txt', sep='\t', header=None)
display(tweets_trainset.head(10))
print(f"Shape: {tweets_trainset.shape}")

,0,1,2
0,1,LOC/Thai Buddhist temple;,"2,000 fetuses found hidden at Thai Buddhist te..."
1,2,LOC/canada;,"870, 000 people in canada depend on #FakeHasht..."
2,3,PER/louis;,"7961212234, phone this girl! she is like louis..."
3,4,ORG/WikiLeaks;LOC/Southern Ocean;,@FakeUsername : WikiLeaks Set To Reveal US-UFO...
4,5,ORG/queen;,"@FakeUsername queen, bohemian rhapsody please"
5,6,PER/cheryl cole;PER/Danni;PER/eddie;,cheryl cole is starting to lose that connectio...
6,7,ORG/Lester;MISC/Mason & Begg Limited;,"Lester, '' Mason & Begg Limited : '' '' `Forbi..."
7,8,PER/Thomas Watson;,"To be successful, you have to have your heart ..."
8,9,"PER/Sanchez;PER/Eli Wallach;MISC/The Good, The...","Sanchez looks like Eli Wallach in The Good, Th..."
9,10,NaN,"@FakeUsername the history of the mayor, the ci..."


Shape: (2815, 3)


In [123]:
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# BIO Tagging Alignment
print("BIO Tagging Alignment\n" + "="*70)

# Helper function: Find entity positions in text
def find_entities(text, entities_dict):
    """Find character positions of all entities in text"""
    entity_spans = {}
    for ent_type, names in entities_dict.items():
        entity_spans[ent_type] = []
        for name in names:
            pattern = re.compile(re.escape(name), re.IGNORECASE)
            for match in pattern.finditer(text):
                entity_spans[ent_type].append((match.start(), match.end()))
    return entity_spans

# Helper function: Parse entity string
def parse_entities(ent_str):
    """Parse 'TYPE/entity; ' format"""
    entities = {}
    if pd.isna(ent_str) or not str(ent_str).strip():
        return entities
    
    for pair in str(ent_str).split(';'):
        if '/' in pair:
            ent_type, name = pair.split('/', 1)
            ent_type = ent_type.strip()
            name = name.strip()
            if ent_type not in entities:
                entities[ent_type] = []
            entities[ent_type].append(name)
    return entities

# Helper function: Assign BIO tags
def get_bio_tags(tokens, positions, entity_spans):
    """Assign BIO tags to tokens based on entity positions"""
    tags = []
    for start, end, token in positions:
        tag = 'O'
        
        for ent_type, spans in entity_spans.items():
            for ent_start, ent_end in spans:
                if start >= ent_start and end <= ent_end:
                    tag = 'B-' + ent_type if start == ent_start else 'I-' + ent_type
                    break
            if tag != 'O':
                break
        tags.append(tag)
    return tags

# Process all sequences
sequences = []
for idx, row in tweets_trainset.iterrows():
    text = row[2]
    
    # Get NLTK tokens with positions
    nltk_tokens = word_tokenize(text)
    tokens_pos = []
    pos = 0
    for token in nltk_tokens:
        start = text.find(token, pos)
        if start == -1:
            start = pos
        end = start + len(token)
        tokens_pos.append((start, end, token))
        pos = end
    
    # Parse and find entities
    entities = parse_entities(row[1])
    entity_spans = find_entities(text, entities)
    
    # Get tokens and tags
    tokens = [t for _, _, t in tokens_pos]
    tags = get_bio_tags(tokens, tokens_pos, entity_spans)
    
    if tokens:
        sequences.append({
            'text': text,
            'tokens': tokens,
            'tags': tags,
            'num_tokens': len(tokens)
        })


sequences_df = pd.DataFrame(sequences)

print(f"\nTotal sequences: {len(sequences_df):,}")

# Sample sequences
print("\n" + "-"*70)
print("SAMPLE SEQUENCES:\n")
for i in range(min(3, len(sequences_df))):
    s = sequences_df.iloc[i]
    print(f"Sequence {i+1}:")
    print(f"  Text:   {s['text']}")
    print(f"  Tokens: {s['tokens']}")
    print(f"  Tags:   {s['tags']}")
    print()

display(sequences_df.head(10))


BIO Tagging Alignment

Total sequences: 2,815

----------------------------------------------------------------------
SAMPLE SEQUENCES:

Sequence 1:
  Text:   2,000 fetuses found hidden at Thai Buddhist temple http://FakeURL via @FakeUsername
  Tokens: ['2,000', 'fetuses', 'found', 'hidden', 'at', 'Thai', 'Buddhist', 'temple', 'http', ':', '//FakeURL', 'via', '@', 'FakeUsername']
  Tags:   ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O']

Sequence 2:
  Text:   870, 000 people in canada depend on #FakeHashtag -25% increase in the last 2 years - please give generously
  Tokens: ['870', ',', '000', 'people', 'in', 'canada', 'depend', 'on', '#', 'FakeHashtag', '-25', '%', 'increase', 'in', 'the', 'last', '2', 'years', '-', 'please', 'give', 'generously']
  Tags:   ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Sequence 3:
  Text:   7961212234, phone this girl! she is like louis biggest fa

,text,tokens,tags,num_tokens
0,"2,000 fetuses found hidden at Thai Buddhist te...","[2,000, fetuses, found, hidden, at, Thai, Budd...","[O, O, O, O, O, B-LOC, I-LOC, I-LOC, O, O, O, ...",14
1,"870, 000 people in canada depend on #FakeHasht...","[870, ,, 000, people, in, canada, depend, on, ...","[O, O, O, O, O, B-LOC, O, O, O, O, O, O, O, O,...",22
2,"7961212234, phone this girl! she is like louis...","[7961212234, ,, phone, this, girl, !, she, is,...","[O, O, O, O, O, O, O, O, O, B-PER, O, O, O, O,...",23
3,@FakeUsername : WikiLeaks Set To Reveal US-UFO...,"[@, FakeUsername, :, WikiLeaks, Set, To, Revea...","[O, O, O, B-ORG, O, O, O, O, O, O, B-LOC, I-LO...",29
4,"@FakeUsername queen, bohemian rhapsody please","[@, FakeUsername, queen, ,, bohemian, rhapsody...","[O, O, B-ORG, O, O, O, O]",7
5,cheryl cole is starting to lose that connectio...,"[cheryl, cole, is, starting, to, lose, that, c...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, B...",28
6,"Lester, '' Mason & Begg Limited : '' '' `Forbi...","[Lester, ,, ``, Mason, &, Begg, Limited, :, ``...","[B-ORG, O, O, B-MISC, I-MISC, I-MISC, I-MISC, ...",33
7,"To be successful, you have to have your heart ...","[To, be, successful, ,, you, have, to, have, y...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",26
8,"Sanchez looks like Eli Wallach in The Good, Th...","[Sanchez, looks, like, Eli, Wallach, in, The, ...","[B-PER, O, O, B-PER, I-PER, O, B-MISC, I-MISC,...",19
9,"@FakeUsername the history of the mayor, the ci...","[@, FakeUsername, the, history, of, the, mayor...","[O, O, O, O, O, O, O, O, O, O, O]",11


In [124]:
from sklearn.model_selection import train_test_split

# Train/Validation Split
print("Data Split - Train/Validation\n" + "="*70)

# Split: 80% train, 20% validation
train_df, val_df = train_test_split(sequences_df, test_size=0.2, random_state=42, shuffle=True)

def df_to_ner_format(df):
    """Convert DataFrame to list of (tokens, tags) for NER training"""
    return [(row['tokens'], row['tags']) for _, row in df.iterrows()]

train_data = df_to_ner_format(train_df)
val_data = df_to_ner_format(val_df)

print(f"Total sequences: {len(sequences_df):,}")
print(f"Training set (80%): {len(train_df):,} sequences")
print(f"Validation set (20%): {len(val_df):,} sequences")

train_true_tags = [row['tags'] for _, row in train_df.iterrows()]
val_true_tags = [row['tags'] for _, row in val_df.iterrows()]

print(f"\nTraining tags: {len(train_true_tags):,} sequences")
print(f"Validation tags: {len(val_true_tags):,} sequences")

#Check
display(train_df.head(10))
sample_tokens, sample_tags = train_data[0]
print("\nSample from Training Set:")
print(f"Tokens: {sample_tokens}")
print(f"Tags:   {sample_tags}")

Data Split - Train/Validation
Total sequences: 2,815
Training set (80%): 2,252 sequences
Validation set (20%): 563 sequences

Training tags: 2,252 sequences
Validation tags: 563 sequences


,text,tokens,tags,num_tokens
1744,Hanie gon get all that Chi Town ass he pull th...,"[Hanie, gon, get, all, that, Chi, Town, ass, h...","[B-PER, O, O, O, O, B-LOC, I-LOC, O, O, O, O, ...",13
2186,Larry Moore recording a late promo ahead of @F...,"[Larry, Moore, recording, a, late, promo, ahea...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O...",17
2371,Natalie Portman : Her Evolution From Eerily Ma...,"[Natalie, Portman, :, Her, Evolution, From, Ee...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O...",24
2514,People are smarter than you think . Give them ...,"[People, are, smarter, than, you, think, ., Gi...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",18
1025,@FakeUsername The fun - and the money - never ...,"[@, FakeUsername, The, fun, -, and, the, money...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",17
2401,Nimrod Laments http://FakeURL,"[Nimrod, Laments, http, :, //FakeURL]","[B-PER, I-PER, O, O, O]",5
644,"@FakeUsername Ima nerd, nerds aint tough right?","[@, FakeUsername, Ima, nerd, ,, nerds, aint, t...","[O, O, B-PER, O, O, O, O, O, O, O]",10
2252,Lol . I thought u missed that RT @FakeUsername...,"[Lol, ., I, thought, u, missed, that, RT, @, F...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",36
461,@FakeUsername #FakeHashtag #FakeHashtag GoI ...,"[@, FakeUsername, #, FakeHashtag, #, FakeHasht...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",31
1868,I commented on a YouTube video http://FakeURL,"[I, commented, on, a, YouTube, video, http, :,...","[O, O, O, O, B-ORG, O, O, O, O]",9



Sample from Training Set:
Tokens: ['Hanie', 'gon', 'get', 'all', 'that', 'Chi', 'Town', 'ass', 'he', 'pull', 'this', 'one', 'off']
Tags:   ['B-PER', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O']


In [125]:
try:
    from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
except ImportError:
    print("seqeval not found.")
    %pip install seqeval

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not found.")
    %pip install matplotlib

# Evaluation function
def evaluate_ner(y_true, y_pred, set_name="Validation"):
    """
    Evaluate NER using seqeval (works directly with BIO tags!)
    
    Args:
        y_true: List of tag sequences [['B-PER', 'I-PER', 'O']]
        y_pred: List of predicted tag sequences
    """
    print(f"\nEvaluation Results for {set_name} Set\n" + "-"*70)

    # Calculate metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Precision (Micro): {precision:.4f}")
    print(f"Recall (Micro):    {recall:.4f}")
    print(f"F1-Score (Micro):  {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {'precision': precision, 'recall': recall, 'f1': f1}

def plot_confusion_matrix(y_true, y_pred, set_name="Validation"):
    """Plot confusion matrix for BIO tags"""
    # Flatten lists
    y_true_flat = [tag for seq in y_true for tag in seq]
    y_pred_flat = [tag for seq in y_pred for tag in seq]

    all_tags = sorted(set(y_true_flat + y_pred_flat))
    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=all_tags)

    fig, ax = plt.subplots(figsize=(12, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels = all_tags)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(f"Confusion Matrix - {set_name} Set")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

print("Evaluation Framework Finished\n")

Evaluation Framework Finished

